In [ ]:
!pip install pandas requests

import requests
import pandas as pd
import time
from google.colab import files


API_KEY = "2bbbed64155396d951fe6035130a9ae2"
BASE_URL = "http://api.aviationstack.com/v1/flights"

AIRPORTS = ["CAI","DXB","JED","IST","DOH","RUH","LHR","CDG"]

all_rows = []

def fetch(params):
    try:
        res = requests.get(BASE_URL, params=params, timeout=20)
        data = res.json()

        if "error" in data:
            print("❌ API Error:", data["error"])
            return []

        return data.get("data", [])

    except Exception as e:
        print("❌ Request Error:", e)
        return []


def extract(f):
    return {
        "Flight Date": f.get("flight_date"),

        "Airline": f.get("airline", {}).get("name"),
        "Airline IATA": f.get("airline", {}).get("iata"),

        "Flight Number": f.get("flight", {}).get("number"),

        "Departure Airport": f.get("departure", {}).get("airport"),
        "Departure IATA": f.get("departure", {}).get("iata"),
        "Departure Scheduled": f.get("departure", {}).get("scheduled"),
        "Departure Actual": f.get("departure", {}).get("actual"),
        "Departure Delay": f.get("departure", {}).get("delay"),

        "Arrival Airport": f.get("arrival", {}).get("airport"),
        "Arrival IATA": f.get("arrival", {}).get("iata"),
        "Arrival Scheduled": f.get("arrival", {}).get("scheduled"),
        "Arrival Actual": f.get("arrival", {}).get("actual"),
        "Arrival Delay": f.get("arrival", {}).get("delay"),

        "Flight Status": f.get("flight_status"),

        "Aircraft": (f.get("aircraft") or {}).get("model")
    }

print("🚀 Start collecting data...")

for airport in AIRPORTS:
    offset = 0

    while True:
        params = {
            "access_key": API_KEY,
            "dep_iata": airport,
            "limit": 100,
            "offset": offset
        }

        flights = fetch(params)

        if not flights:
            print(f"⛔ No more data for {airport}")
            break

        for f in flights:
            row = extract(f)
            if any(row.values()):
                all_rows.append(row)

        print(f"✈️ {airport} → Total rows: {len(all_rows)}")

        offset += 100
        time.sleep(2)

df = pd.DataFrame(all_rows)


df.drop_duplicates(inplace=True)

df.dropna(how="all", inplace=True)


df = df[(df["Departure Scheduled"].notna()) | (df["Arrival Scheduled"].notna())]

try:
    df["Departure Hour"] = pd.to_datetime(df["Departure Scheduled"]).dt.hour
    df["Day of Week"] = pd.to_datetime(df["Flight Date"]).dt.day_name()
except:
    pass

df["Is Delayed"] = df["Departure Delay"].fillna(0) > 15

df["Route"] = df["Departure IATA"].astype(str) + " → " + df["Arrival IATA"].astype(str)


print("\n📊 Final Dataset Size:", len(df))
print(df.head())

if len(df) > 0:
    file_name = "aviationstack_flights_clean.csv"
    df.to_csv(file_name, index=False)

    print("✅ File saved successfully!")

    files.download(file_name)

else:
    print("❌ No data collected")

print("DONE ✅")

🚀 Start collecting data...
✈️ CAI → Total rows: 100
✈️ CAI → Total rows: 200
✈️ CAI → Total rows: 300
✈️ CAI → Total rows: 400
✈️ CAI → Total rows: 500
✈️ CAI → Total rows: 600
✈️ CAI → Total rows: 700
✈️ CAI → Total rows: 800
✈️ CAI → Total rows: 900
✈️ CAI → Total rows: 989
⛔ No more data for CAI
✈️ DXB → Total rows: 1089
✈️ DXB → Total rows: 1189
✈️ DXB → Total rows: 1289
✈️ DXB → Total rows: 1389
✈️ DXB → Total rows: 1489
✈️ DXB → Total rows: 1589
✈️ DXB → Total rows: 1689
✈️ DXB → Total rows: 1789
✈️ DXB → Total rows: 1889
✈️ DXB → Total rows: 1989
✈️ DXB → Total rows: 2089
✈️ DXB → Total rows: 2189
✈️ DXB → Total rows: 2289
✈️ DXB → Total rows: 2344
⛔ No more data for DXB
✈️ JED → Total rows: 2444
✈️ JED → Total rows: 2544
✈️ JED → Total rows: 2644
✈️ JED → Total rows: 2744
✈️ JED → Total rows: 2844
✈️ JED → Total rows: 2944
✈️ JED → Total rows: 3044
✈️ JED → Total rows: 3144
✈️ JED → Total rows: 3244
✈️ JED → Total rows: 3344
✈️ JED → Total rows: 3444
✈️ JED → Total rows: 3506
⛔

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DONE ✅
